# 02 — Demand EDA, series selection and forecast error analysis

Data: `data/gold/ml/demand_daily.parquet` (daily distinct revenue orders per
product_category × customer_state, zero-filled from each series' first order to E, the last day
before the extract's truncated tail). Model: `demand_forecast@champion` on the MLflow server.

Run top-to-bottom from anywhere inside the repo, e.g.
`uv run --project ml --with jupyter --with nbconvert --with matplotlib jupyter nbconvert --execute --to notebook --inplace ds/notebooks/02_demand_eda.ipynb`.
Outputs are cleared before commit; conclusions live in `ds/reports/` and `ds/model_cards/`.

In [ ]:
import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
os.chdir(ROOT)
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
# presigned MinIO URLs (minio:9000) only resolve inside Compose; stream via the tracking server
os.environ.setdefault("MLFLOW_ENABLE_PROXY_MULTIPART_DOWNLOAD", "false")

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from retail_ml.forecast.features import HORIZON, KEYS, modelled_series, seasonal_naive, wape, window
from retail_ml.forecast.train import load_demand_forecast_config, model_inputs, read_demand

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
TAB = ROOT / "ds/reports/tables"
TAB.mkdir(parents=True, exist_ok=True)
cfg = load_demand_forecast_config()
client = MlflowClient()

## 1. Coverage, series selection and the truncated tail

In [ ]:
full = read_demand(cfg.demand_path)
df, excluded = modelled_series(full)
df = df.reset_index(drop=True)
per_series = full.groupby(KEYS).agg(
    orders=("orders", "sum"), days=("orders", "size"), modelled=("is_modelled", "first")
)
print("dates", full["date"].min().date(), "->", full["date"].max().date())
print("series", len(per_series), "modelled", int(per_series["modelled"].sum()))
print(
    "share of series-orders in modelled series (whole span)",
    round(per_series.loc[per_series["modelled"], "orders"].sum() / per_series["orders"].sum(), 4),
)
end = df["date"].max()
cutoff = end - pd.Timedelta(days=HORIZON - 1)
win = full[(full["date"] >= cutoff - pd.Timedelta(days=180)) & (full["date"] < cutoff)]
print(
    "share in the 180-day selection window",
    round(win.loc[win["is_modelled"], "orders"].sum() / win["orders"].sum(), 4),
)
print("test window", cutoff.date(), "->", end.date())

In [ ]:
raw = pd.read_parquet(
    ROOT / "data/gold/ml/late_delivery_training.parquet",
    columns=["order_id", "order_purchase_ts_utc"],
)
purchases = raw.groupby(raw["order_purchase_ts_utc"].dt.tz_localize(None).dt.floor("D")).size()
print("approved orders per purchase day around E (from the late-delivery contract):")
print(purchases.loc[end - pd.Timedelta(days=5) : end + pd.Timedelta(days=10)].to_string())

## 2. Seasonality (all series)

In [ ]:
total = full.groupby("date")["orders"].sum()
fig, ax = plt.subplots(figsize=(11, 3))
total.plot(ax=ax, lw=0.7)
total.rolling(28).mean().plot(ax=ax)
ax.axvline(cutoff, color="grey", ls="--")
ax.set_title("Daily series-orders, all series (28-day mean; dashed = test start)")
plt.show()
recent = full[full["date"] >= "2017-06-01"]
by_dow = recent.groupby(recent["date"].dt.dayofweek)["orders"].sum()
print("weekday share (0=Mon) since 2017-06:", (by_dow / by_dow.sum()).round(4).to_dict())
monthly = full.groupby(full["date"].dt.to_period("M"))["orders"].sum()
monthly.tail(14)

## 3. Sparsity: modelled vs excluded series (180-day selection window)

In [ ]:
w = win.groupby(KEYS).agg(
    mean_daily=("orders", "mean"),
    active_share=("orders", lambda s: (s > 0).mean()),
    modelled=("is_modelled", "first"),
)
print(w.groupby("modelled")[["mean_daily", "active_share"]].describe().T)
by_state = df.groupby("customer_state")[KEYS[0]].nunique().sort_values(ascending=False)
print("modelled series by state:", by_state.to_dict())

## 4. Test window: champion vs seasonal naive, per series

In [ ]:
version = client.get_model_version_by_alias(cfg.registered_model, "champion")
champ_run = client.get_run(version.run_id)
exp = client.get_experiment_by_name(cfg.experiment)
runs = {
    r.info.run_name: r
    for r in client.search_runs([exp.experiment_id], order_by=["start_time ASC"])
    if r.data.tags.get("data_sha256") == champ_run.data.tags["data_sha256"]
}
champion = mlflow.pyfunc.load_model(f"models:/{cfg.registered_model}@champion")
frame = window(df, cutoff)
horizon = frame.index[frame["date"] >= cutoff]
p = np.asarray(champion.predict(model_inputs(frame)), dtype=float)
test = df.loc[horizon, ["date", *KEYS]].assign(
    actual=df.loc[horizon, "orders"],
    lightgbm=p[frame.index.get_indexer(horizon)],
    naive=seasonal_naive(frame, cutoff).loc[horizon].to_numpy(),
)
overall = {m: wape(test["actual"], test[m]) for m in ("naive", "lightgbm")}
logged = {
    "naive": runs["seasonal_naive"].data.metrics["test_wape"],
    "lightgbm": runs["lightgbm"].data.metrics["test_wape"],
}
print(f"champion v{version.version}; recomputed test WAPE", overall, "MLflow", logged)
assert all(abs(overall[m] - logged[m]) < 1e-6 for m in overall)
print(
    "bias (sum forecast / sum actual):",
    {m: round(test[m].sum() / test["actual"].sum(), 4) for m in ("naive", "lightgbm")},
)

In [ ]:
history = df[df["date"] < cutoff]
vol = history.groupby(KEYS)["orders"].sum().rename("history_orders")
prev = history[history["date"] >= cutoff - pd.Timedelta(days=HORIZON)]
last28 = prev.groupby(KEYS)["orders"].mean()
series = test.groupby(KEYS).apply(
    lambda g: pd.Series(
        {
            "test_orders": g["actual"].sum(),
            "wape_naive": wape(g["actual"], g["naive"]),
            "wape_lightgbm": wape(g["actual"], g["lightgbm"]),
            "test_zero_share": (g["actual"] == 0).mean(),
            "test_mean": g["actual"].mean(),
        }
    ),
    include_groups=False,
)
series = series.join(vol).join(last28.rename("prev28_mean"))
series["level_shift"] = series["test_mean"] / series["prev28_mean"]
series["delta"] = series["wape_lightgbm"] - series["wape_naive"]
series["lightgbm_loses"] = series["delta"] > 0
series = series.sort_values("history_orders", ascending=False)
series.to_csv(TAB / "demand_test_per_series.csv")
top10 = series.head(cfg.top_series)
for (cat, state), r in top10.iterrows():
    key = f"test_wape.{cat}.{state}"
    assert abs(r["wape_lightgbm"] - runs["lightgbm"].data.metrics[key]) < 1e-6, key
top10

In [ ]:
loses = series[series["lightgbm_loses"]]
print(
    "series where LightGBM WAPE > naive:",
    len(loses),
    "of",
    len(series),
    "| share of test orders:",
    round(loses["test_orders"].sum() / series["test_orders"].sum(), 4),
)
cols = ["test_orders", "test_zero_share", "level_shift", "wape_naive", "wape_lightgbm"]
print(series.groupby("lightgbm_loses")[cols].median())
loses.sort_values("test_orders", ascending=False)[cols + ["delta"]].head(10)

In [ ]:
vb = pd.qcut(series["history_orders"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
quartiles = test.merge(vb.rename("volume_quartile").reset_index(), on=KEYS)
by_vol = quartiles.groupby("volume_quartile", observed=True).apply(
    lambda g: pd.Series(
        {
            "series": g[KEYS].drop_duplicates().shape[0],
            "test_orders": g["actual"].sum(),
            "wape_naive": wape(g["actual"], g["naive"]),
            "wape_lightgbm": wape(g["actual"], g["lightgbm"]),
        }
    ),
    include_groups=False,
)
by_vol.to_csv(TAB / "demand_test_by_volume_quartile.csv")
by_h = (
    test.assign(week=((test["date"] - cutoff).dt.days // 7) + 1)
    .groupby("week")
    .apply(
        lambda g: pd.Series(
            {
                "wape_naive": wape(g["actual"], g["naive"]),
                "wape_lightgbm": wape(g["actual"], g["lightgbm"]),
            }
        ),
        include_groups=False,
    )
)
by_h.to_csv(TAB / "demand_test_by_horizon_week.csv")
print(by_vol)
by_h

Daily totals over the test window: the last days of the extract ramp down (see section 1), so
week-4 actuals sit below normal for both models.

In [ ]:
daily = test.groupby("date")[["actual", "naive", "lightgbm"]].sum()
daily.to_csv(TAB / "demand_test_daily_totals.csv")
daily.tail(10)